# VisionCart — Online Evaluation (Google Colab)

This notebook handles everything that needs a GPU:
1. **CLIP embeddings** — visual similarity between vision board images and product images
2. **Style personas** — real Qwen2.5-VL-7B output from vision board images

After this notebook runs, `eval.py` picks up the saved files automatically:
- `data/embeddings/board_embeddings.json` — one CLIP vector per style
- `data/embeddings/product_embeddings.json` — one CLIP vector per product
- `data/style_personas.json` — one VLM-generated style profile per style

**Offline eval** (no GPU, STYLE_DEFINITIONS only):  
`python eval.py --split validation --variant all`

**Online eval** (VLM personas + CLIP embeddings):  
`python eval.py --split validation --variant all --personas data/style_personas.json`

## 1. Check GPU

In [1]:
import torch
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime > Change runtime type > GPU (T4 recommended)')

GPU : NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Install Dependencies

In [2]:
%%capture
!pip install transformers>=4.45 accelerate bitsandbytes Pillow python-dotenv anthropic langgraph

## 3. Mount Repo

Choose one option:

In [3]:
import os, sys

# ── Option A: Google Drive ────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# REPO = '/content/drive/MyDrive/VisionCart'   # adjust to your path

# ── Option B: Clone from GitHub ───────────────────────────────────────────────
# !git clone https://github.com/YOUR_USERNAME/VisionCart.git /content/VisionCart
# REPO = '/content/VisionCart'

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/Coursework/Spring2026/INFO290/genai-final-project/VisionCart/'   # set this to whichever option you used
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, f'{REPO}/src')
print('Working dir:', os.getcwd())
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/MyDrive/Coursework/Spring2026/INFO290/genai-final-project/VisionCart
data	 docs	  notebooks    README.md	 results  tests
dataset  eval.py  __pycache__  requirements.txt  src	  venv


## 4. API Tokens

Add `HF_TOKEN` (and optionally `ANTHROPIC_API_KEY`) to the 🔑 **Colab Secrets** sidebar before running.

In [4]:
import os
import getpass
from dotenv import load_dotenv

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded.')
except Exception:
    print('HF_TOKEN not found in Colab Secrets — set manually if needed.')


# Run this cell; a text box will appear at the top of VS Code
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF_TOKEN here:")

print("Keys set for this remote session.")

HF_TOKEN not found in Colab Secrets — set manually if needed.
Keys set for this remote session.


In [5]:
# 1. Check API Keys
load_dotenv()
print(f"HF_TOKEN present: {bool(os.environ.get('HF_TOKEN'))}")

HF_TOKEN present: True


In [6]:
from huggingface_hub import notebook_login, whoami
try:
    print(f"Authenticated as: {whoami()['name']}")
except Exception:
    print("Authentication failed. Check your token.")

Authenticated as: kskaneko


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


---
## 5. CLIP — Board Embeddings

Embeds the vision board images in `dataset/images/<style_label>/` using  
`openai/clip-vit-base-patch32` (512-dim).  One mean vector per style.

In [ ]:
from pathlib import Path
from utils.clip_embeddings import (
    compute_board_embeddings,
    load_board_embeddings,
    save_embeddings,
    BOARD_EMBEDDINGS_PATH,
)

IMAGES_BASE = Path('dataset/images')

# Discover all style folders that have images
style_labels = [
    d.name for d in sorted(IMAGES_BASE.iterdir())
    if d.is_dir() and any(f.suffix.lower() in {'.jpg','.jpeg','.png'} for f in d.iterdir())
] if IMAGES_BASE.exists() else []

print(f'Styles with images: {style_labels}')

board_embeddings = compute_board_embeddings(IMAGES_BASE, style_labels)
save_embeddings(board_embeddings, BOARD_EMBEDDINGS_PATH)
print(f'\nDone. {len(board_embeddings)}/{len(style_labels)} styles embedded.')

## 6. CLIP — Product Embeddings

Embeds every unique product image in `dataset/dataset.json`.  
Uses `image_local` (local file) when available, otherwise downloads `image_url`.

**This cell can take a few minutes.** Products with no accessible image are skipped.

In [ ]:
from utils.clip_embeddings import (
    compute_product_embeddings,
    save_embeddings,
    PRODUCT_EMBEDDINGS_PATH,
)

DATASET_PATH = Path('dataset/dataset.json')

product_embeddings = compute_product_embeddings(DATASET_PATH, prefer_local=True)
save_embeddings(product_embeddings, PRODUCT_EMBEDDINGS_PATH)
print(f'\nDone. {len(product_embeddings)} products embedded.')

---
## 7. Generate Style Personas (Qwen2.5-VL-7B)

Runs the real stylist VLM **once per style** on its vision board images.  
Saves ranker-compatible profiles to `data/style_personas.json`.

**First run downloads ~15 GB of model weights — takes ~5 min on T4.**

In [ ]:
import json
from pathlib import Path
from agents import stylist as _stylist

IMAGES_BASE   = Path('dataset/images')
PERSONAS_PATH = Path('data/style_personas.json')

# Load board embeddings so they're included in the saved personas
from utils.clip_embeddings import load_board_embeddings, BOARD_EMBEDDINGS_PATH
board_embeddings = load_board_embeddings(BOARD_EMBEDDINGS_PATH)


def stylist_output_to_profile(style_label, stylist_out, board_emb):
    """Convert raw stylist output to a ranker-compatible style_profile dict."""
    aesthetic = stylist_out.get('aesthetic')  or []
    colors    = stylist_out.get('colors')     or []
    materials = stylist_out.get('materials')  or []
    products  = stylist_out.get('products')   or []
    narrative = stylist_out.get('style_profile', '')
    product_tokens = [tok for phrase in products for tok in phrase.lower().split()]
    style_keywords = list(dict.fromkeys(aesthetic + colors + materials + product_tokens))
    return {
        'board_id':        style_label,
        'style_summary':   narrative,
        'style_keywords':  style_keywords,
        'style_elements':  aesthetic,
        'color_palette':   {'dominant': colors, 'accent': [], 'avoid': []},
        'materials':       {'preferred': materials, 'avoid': []},
        'constraints':     {'must_avoid': []},
        'board_embedding': board_emb.get(style_label, []),
    }


personas = {}

style_dirs = sorted(
    d for d in IMAGES_BASE.iterdir()
    if d.is_dir() and any(f.suffix.lower() in {'.jpg','.jpeg','.png'} for f in d.iterdir())
) if IMAGES_BASE.exists() else []

for style_dir in style_dirs:
    style_label = style_dir.name
    image_paths = sorted(
        str(f) for f in style_dir.iterdir()
        if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    )[:5]

    print(f'\n[stylist] {style_label} ({len(image_paths)} images) …')
    result = _stylist.run({'vision_board_paths': image_paths})
    stylist_out = result.get('stylist_output', {})
    profile = stylist_output_to_profile(style_label, stylist_out, board_embeddings)
    personas[style_label] = profile
    print(f'  keywords : {profile["style_keywords"][:6]}')
    print(f'  summary  : {profile["style_summary"][:80]} …')

PERSONAS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(PERSONAS_PATH, 'w') as f:
    json.dump(personas, f, indent=2)
print(f'\nSaved {len(personas)} personas → {PERSONAS_PATH}')

---
## 8. Sanity Check — What Was Generated

In [ ]:
import json
from pathlib import Path

board_emb  = json.loads(Path('data/embeddings/board_embeddings.json').read_text()) if Path('data/embeddings/board_embeddings.json').exists() else {}
prod_emb   = json.loads(Path('data/embeddings/product_embeddings.json').read_text()) if Path('data/embeddings/product_embeddings.json').exists() else {}
personas   = json.loads(Path('data/style_personas.json').read_text()) if Path('data/style_personas.json').exists() else {}

print(f'Board embeddings  : {len(board_emb)} styles  (dim={len(next(iter(board_emb.values()), []))})')
print(f'Product embeddings: {len(prod_emb)} products (dim={len(next(iter(prod_emb.values()), []))})')
print(f'Style personas    : {len(personas)} styles')
print()
for style, p in list(personas.items())[:3]:
    has_emb = bool(p.get('board_embedding'))
    print(f'  {style:<30} keywords={p["style_keywords"][:4]}  board_embedding={'yes' if has_emb else 'NO'}')

---
## 9. Run Offline Eval

Uses STYLE_DEFINITIONS personas + pre-computed CLIP embeddings.  
This is the baseline — static keywords with visual similarity.

In [7]:
!python eval.py --split validation --variant all --k 5 --out results/offline_results.json --verbose

[eval] Loaded board embeddings for 10 styles.
[eval] Loaded product embeddings for 120 products.

Running variant: BM25  |  split: validation  |  entries: 24
  [bm25] dark_academia                | herringbone wool blazer women                  MRR=0.25  P@5=0.20  R_full=1.00
  [bm25] dark_academia                | leather satchel bag vintage brown              MRR=0.11  P@5=0.00  R_full=1.00
  [bm25] dark_academia                | high waist tweed trousers                      MRR=0.08  P@5=0.00  R_full=1.00
  [bm25] quiet_luxury                 | camel cashmere turtleneck sweater              MRR=0.08  P@5=0.00  R_full=1.00
  [bm25] quiet_luxury                 | straight leg tailored trousers beige           MRR=0.06  P@5=0.00  R_full=1.00
  [bm25] quiet_luxury                 | leather loafer minimal                         MRR=0.05  P@5=0.00  R_full=1.00
  [bm25] y2k_streetwear               | rhinestone butterfly graphic baby tee          MRR=0.02  P@5=0.00  R_full=1.00
  [bm25] 

## 10. Run Online Eval

Uses real VLM personas + pre-computed CLIP embeddings.  
BM25 is unaffected (always uses style label only).

In [8]:
!python eval.py --split validation --variant all --k 5 --personas data/style_personas.json --out results/online_results.json --verbose

[eval] Loaded board embeddings for 10 styles.
[eval] Loaded product embeddings for 120 products.
[eval] Loaded 10 real style personas from data/style_personas.json.

Running variant: BM25  |  split: validation  |  entries: 24
  [bm25] dark_academia                | herringbone wool blazer women                  MRR=0.25  P@5=0.20  R_full=1.00
  [bm25] dark_academia                | leather satchel bag vintage brown              MRR=0.11  P@5=0.00  R_full=1.00
  [bm25] dark_academia                | high waist tweed trousers                      MRR=0.08  P@5=0.00  R_full=1.00
  [bm25] quiet_luxury                 | camel cashmere turtleneck sweater              MRR=0.08  P@5=0.00  R_full=1.00
  [bm25] quiet_luxury                 | straight leg tailored trousers beige           MRR=0.06  P@5=0.00  R_full=1.00
  [bm25] quiet_luxury                 | leather loafer minimal                         MRR=0.05  P@5=0.00  R_full=1.00
  [bm25] y2k_streetwear               | rhinestone butterfly

---
## 11. Compare Results

In [9]:
import json
from pathlib import Path

offline = json.loads(Path('results/offline_results.json').read_text())
online  = json.loads(Path('results/online_results.json').read_text())

variants = ['bm25', 'lc', 'lg']
metrics  = [
    ('MRR',          'mean_mrr'),
    ('Precision@5',  'mean_precision_at_5'),
    ('Recall@5',     'mean_recall_at_5'),
    ('Recall (full)','mean_recall_full'),
]

for metric_label, metric_key in metrics:
    print(f'\n{metric_label}')
    print(f"  {'Variant':<8} {'Offline':>10} {'Online':>10} {'Delta':>10}")
    print('  ' + '-'*40)
    for v in variants:
        off = offline['results'][v]['aggregate'].get(metric_key) or 0
        on  = online['results'][v]['aggregate'].get(metric_key) or 0
        d   = on - off
        sign = '+' if d >= 0 else ''
        print(f"  {v.upper():<8} {off:>10.4f} {on:>10.4f} {sign}{d:>9.4f}")


MRR
  Variant     Offline     Online      Delta
  ----------------------------------------
  BM25         0.1984     0.1984 +   0.0000
  LC           0.4573     0.4445   -0.0128
  LG           0.4573     0.4445   -0.0128

Precision@5
  Variant     Offline     Online      Delta
  ----------------------------------------
  BM25         0.1000     0.1000 +   0.0000
  LC           0.3083     0.2667   -0.0416
  LG           0.3083     0.2667   -0.0416

Recall@5
  Variant     Offline     Online      Delta
  ----------------------------------------
  BM25         0.1250     0.1250 +   0.0000
  LC           0.3854     0.3333   -0.0521
  LG           0.3854     0.3333   -0.0521

Recall (full)
  Variant     Offline     Online      Delta
  ----------------------------------------
  BM25         1.0000     1.0000 +   0.0000
  LC           1.0000     1.0000 +   0.0000
  LG           1.0000     1.0000 +   0.0000


## 12. Full Eval Log

In [10]:
from IPython.display import Markdown, display
from pathlib import Path

log = Path('results/eval_log.md')
display(Markdown(log.read_text())) if log.exists() else print('No log yet.')

## 2026-04-27 01:10:03 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | style: dark_academia | queries: 3

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.3333 | 0.3333 | 0.3333 |
| Precision@3 | 0.3333 | 0.3333 | 0.3333 |
| Precision@5 | 0.3333 | 0.3333 | 0.3333 |
| Recall@1 | 0.0833 | 0.0833 | 0.0833 |
| Recall@3 | 0.2500 | 0.2500 | 0.2500 |
| Recall@5 | 0.4167 | 0.4167 | 0.4167 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.4861 | 0.4815 | 0.5833 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| dark_academia | 0.4861 | 0.4815 | 0.5833 |

---
## 2026-04-27 01:10:09 — test_eval.py | mode: ranker (full pool) | style: dark_academia | queries: 3

| Metric | Value |
|--------|-------|
| Total queries | 3 |
| Perfect recall | 3 / 3 |
| Mean recall | 1.0000 |

**Recall by style**

| Style | Recall |
|-------|--------|
| dark_academia | 1.0000 |

---
## 2026-04-27 01:17:22 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.3333 | 0.3333 | 0.2500 |
| Precision@3 | 0.3333 | 0.3333 | 0.2778 |
| Precision@5 | 0.3333 | 0.3333 | 0.2500 |
| Recall@1 | 0.0833 | 0.0833 | 0.0625 |
| Recall@3 | 0.2500 | 0.2500 | 0.2083 |
| Recall@5 | 0.4167 | 0.4167 | 0.3125 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.5304 | 0.4518 | 0.4214 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.5476 | 0.4370 | 0.3611 |
| dark_academia | 0.4861 | 0.4815 | 0.5833 |
| dark_moody_organic | 0.6111 | 0.4370 | 0.3111 |
| maximalist_eclectic | 0.4643 | 0.4370 | 0.4087 |
| mid_century_modern | 0.4921 | 0.4370 | 0.4256 |
| quiet_luxury | 0.5833 | 0.4104 | 0.3568 |
| vintage_bohemian | 0.5111 | 0.5370 | 0.5370 |
| y2k_streetwear | 0.5476 | 0.4370 | 0.3876 |

---
## 2026-04-28 08:14:48 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | personas: real | clip: no | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.3333 | 0.3333 | 0.3333 |
| Precision@3 | 0.3333 | 0.3056 | 0.3056 |
| Precision@5 | 0.3333 | 0.2917 | 0.2917 |
| Recall@1 | 0.0833 | 0.0833 | 0.0833 |
| Recall@3 | 0.2500 | 0.2292 | 0.2292 |
| Recall@5 | 0.4167 | 0.3646 | 0.3646 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.4809 | 0.4703 | 0.4703 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.4537 | 0.5370 | 0.5370 |
| dark_academia | 0.4861 | 0.4537 | 0.4537 |
| dark_moody_organic | 0.4417 | 0.5000 | 0.5000 |
| maximalist_eclectic | 0.4643 | 0.3905 | 0.3905 |
| mid_century_modern | 0.5000 | 0.5476 | 0.5476 |
| quiet_luxury | 0.5833 | 0.4500 | 0.4500 |
| vintage_bohemian | 0.4815 | 0.4417 | 0.4417 |
| y2k_streetwear | 0.4370 | 0.4417 | 0.4417 |

---
## 2026-04-28 08:15:24 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | clip: no | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.3333 | 0.3333 | 0.3333 |
| Precision@3 | 0.3333 | 0.3333 | 0.3333 |
| Precision@5 | 0.3333 | 0.3333 | 0.3333 |
| Recall@1 | 0.0833 | 0.0833 | 0.0833 |
| Recall@3 | 0.2500 | 0.2500 | 0.2500 |
| Recall@5 | 0.4167 | 0.4167 | 0.4167 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.4809 | 0.4969 | 0.4969 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.4537 | 0.4815 | 0.4815 |
| dark_academia | 0.4861 | 0.4583 | 0.4583 |
| dark_moody_organic | 0.4417 | 0.6111 | 0.6111 |
| maximalist_eclectic | 0.4643 | 0.4370 | 0.4370 |
| mid_century_modern | 0.5000 | 0.4643 | 0.4643 |
| quiet_luxury | 0.5833 | 0.4333 | 0.4333 |
| vintage_bohemian | 0.4815 | 0.5476 | 0.5476 |
| y2k_streetwear | 0.4370 | 0.5417 | 0.5417 |

---
## 2026-04-28 08:40:55 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | clip: yes | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.3333 | 0.3333 | 0.3333 |
| Precision@3 | 0.3333 | 0.3333 | 0.3333 |
| Precision@5 | 0.3333 | 0.3333 | 0.3333 |
| Recall@1 | 0.0833 | 0.0833 | 0.0833 |
| Recall@3 | 0.2500 | 0.2500 | 0.2500 |
| Recall@5 | 0.4167 | 0.4167 | 0.4167 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.4789 | 0.4969 | 0.4969 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.4370 | 0.4815 | 0.4815 |
| dark_academia | 0.4861 | 0.4583 | 0.4583 |
| dark_moody_organic | 0.4417 | 0.6111 | 0.6111 |
| maximalist_eclectic | 0.4643 | 0.4370 | 0.4370 |
| mid_century_modern | 0.5000 | 0.4643 | 0.4643 |
| quiet_luxury | 0.5833 | 0.4333 | 0.4333 |
| vintage_bohemian | 0.4815 | 0.5476 | 0.5476 |
| y2k_streetwear | 0.4370 | 0.5417 | 0.5417 |

---
## 2026-04-28 08:41:01 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | personas: real | clip: yes | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.3333 | 0.3333 | 0.3333 |
| Precision@3 | 0.3333 | 0.3056 | 0.3056 |
| Precision@5 | 0.3333 | 0.2917 | 0.2917 |
| Recall@1 | 0.0833 | 0.0833 | 0.0833 |
| Recall@3 | 0.2500 | 0.2292 | 0.2292 |
| Recall@5 | 0.4167 | 0.3646 | 0.3646 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.4789 | 0.4764 | 0.4764 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.4370 | 0.5303 | 0.5303 |
| dark_academia | 0.4861 | 0.4537 | 0.4537 |
| dark_moody_organic | 0.4417 | 0.5556 | 0.5556 |
| maximalist_eclectic | 0.4643 | 0.3905 | 0.3905 |
| mid_century_modern | 0.5000 | 0.5476 | 0.5476 |
| quiet_luxury | 0.5833 | 0.4500 | 0.4500 |
| vintage_bohemian | 0.4815 | 0.4417 | 0.4417 |
| y2k_streetwear | 0.4370 | 0.4417 | 0.4417 |

---
## 2026-04-28 08:45:14 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | clip: yes | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.1250 | 0.3333 | 0.3333 |
| Precision@3 | 0.1250 | 0.3194 | 0.3194 |
| Precision@5 | 0.1000 | 0.3083 | 0.3083 |
| Recall@1 | 0.0312 | 0.0833 | 0.0833 |
| Recall@3 | 0.0938 | 0.2396 | 0.2396 |
| Recall@5 | 0.1250 | 0.3854 | 0.3854 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.1984 | 0.4573 | 0.4573 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.0113 | 0.4444 | 0.4444 |
| dark_academia | 0.1460 | 0.4583 | 0.4583 |
| dark_moody_organic | 0.1724 | 0.4226 | 0.4226 |
| maximalist_eclectic | 0.0099 | 0.4333 | 0.4333 |
| mid_century_modern | 0.4921 | 0.4583 | 0.4583 |
| quiet_luxury | 0.0611 | 0.4333 | 0.4333 |
| vintage_bohemian | 0.3458 | 0.5303 | 0.5303 |
| y2k_streetwear | 0.3489 | 0.4778 | 0.4778 |

---
## 2026-04-28 08:45:22 — eval.py | split: validation | variants: bm25, lc, lg | k=5 | personas: real | clip: yes | queries: 24

| Metric | BM25 | LC | LG |
|--------|--------|--------|--------|
| Precision@1 | 0.1250 | 0.2917 | 0.2917 |
| Precision@3 | 0.1250 | 0.2778 | 0.2778 |
| Precision@5 | 0.1000 | 0.2667 | 0.2667 |
| Recall@1 | 0.0312 | 0.0729 | 0.0729 |
| Recall@3 | 0.0938 | 0.2083 | 0.2083 |
| Recall@5 | 0.1250 | 0.3333 | 0.3333 |
| Recall (full) | 1.0000 | 1.0000 | 1.0000 |
| MRR | 0.1984 | 0.4445 | 0.4445 |
| LLM Aesth. | N/A | N/A | N/A |
| LLM Expl. | N/A | N/A | N/A |

**MRR by style**

| Style | BM25 | LC | LG |
|-------|--------|--------|--------|
| coastal_mediterranean | 0.0113 | 0.5303 | 0.5303 |
| dark_academia | 0.1460 | 0.4537 | 0.4537 |
| dark_moody_organic | 0.1724 | 0.5000 | 0.5000 |
| maximalist_eclectic | 0.0099 | 0.3822 | 0.3822 |
| mid_century_modern | 0.4921 | 0.5667 | 0.5667 |
| quiet_luxury | 0.0611 | 0.4470 | 0.4470 |
| vintage_bohemian | 0.3458 | 0.4167 | 0.4167 |
| y2k_streetwear | 0.3489 | 0.2593 | 0.2593 |

---
